# 📈 Three-Candle Pivot Breakout: Backtest Analysis
This notebook allows you to run historical backtests, analyze performance metrics, and visualize PnL curves for your trading strategy.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import logging

# Ensure project root is in path for imports
sys.path.append(os.path.abspath('..'))

from backtest.engine import BacktestEngine
from bot.strategy import ThreeCandleV2Live
from backtest.metrics import calculate_metrics

### 1. Configuration
Set your backtest parameters here.

In [ ]:
CONFIG = {
    'instrument': 'NSE_INDEX|Nifty 50',
    'csv_path': '../data/csv/NIFTY50_15min_6months.csv',
    'pivot_levels': [23800, 23900, 24000, 24100, 24200, 24300, 24400, 24500, 24600],
    'lot_size': 25
}

### 2. Run Backtest
Execute the event-driven simulation.

In [ ]:
engine = BacktestEngine(
    strategy_class=ThreeCandleV2Live,
    instrument_key=CONFIG['instrument'],
    csv_path=CONFIG['csv_path'],
    pivot_levels=CONFIG['pivot_levels']
)

# Override lot size if needed
engine.strategy.risk.lot_size = CONFIG['lot_size']
engine.executor.risk.lot_size = CONFIG['lot_size']

await engine.run()

# Display Metrics
print(f"Total Trades   : {engine.executor.state.total_trades}")
print(f"Net Realized PnL: Rs. {engine.executor.state.realized_pnl * CONFIG['lot_size']:,.2f}")

### 3. PnL Visualization
Visualize the growth of your account over time.

In [ ]:
trade_history = engine.executor.state.trade_history
if trade_history:
    df_trades = pd.DataFrame(trade_history)
    df_trades['cumulative_pnl'] = df_trades['pnl'].cumsum() * CONFIG['lot_size']
    
    plt.figure(figsize=(12, 6))
    plt.plot(df_trades['cumulative_pnl'], marker='o', linestyle='-', color='blue')
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Cumulative PnL (INR)')
    plt.xlabel('Trade Number')
    plt.ylabel('PnL (Rs.)')
    plt.grid(True)
    plt.show()
else:
    print("No trades to display.")